# Tugas Praktikum 5 — Primitif Sinkronisasi
**Nama:** Virgi Athaila Akbar  
**NIM:** 2411513020  

Notebook ini membahas tiga eksperimen: penggunaan `RLock` pada rekening bank, simulasi deadlock dengan tiga lock dan tiga thread, serta bounded buffer dengan kapasitas berbeda dan dua producer–dua consumer.

> Jalankan sel secara berurutan. Hasil waktu dan urutan interleaving dapat sedikit berbeda pada setiap eksekusi karena penjadwalan thread oleh sistem operasi.

## 1. RLock pada Class `RekeningBank`

### Tujuan
Membuat rekening bank dengan method `setor(jumlah)`, `tarik(jumlah)`, dan `transfer(rekening_tujuan, jumlah)`. Setiap objek memiliki `RLock` sendiri. Method `transfer()` memanggil `tarik()` pada rekening sumber dan `setor()` pada rekening tujuan.

### Alasan menggunakan `RLock`
Pada saat `transfer()` mengunci rekening sumber, method tersebut kemudian memanggil `tarik()` yang juga berusaha mengunci objek rekening sumber. `RLock` mengizinkan thread yang sama mengambil lock yang sama berulang kali (reentrant). Jika menggunakan `Lock` biasa, penguncian kedua oleh thread yang sama tidak dapat dilanjutkan sehingga terjadi self-deadlock.

Untuk menghindari deadlock ketika dua transfer berlangsung berlawanan arah (A ke B dan B ke A), kedua lock rekening diambil dengan urutan konsisten berdasarkan identitas objek. Pemanggilan `tarik()` dan `setor()` tetap dilakukan di dalam bagian yang terkunci.

In [1]:
from threading import RLock, Thread
from contextlib import ExitStack
import time

class RekeningBank:
    def __init__(self, nama, saldo=0):
        self.nama = nama
        self.saldo = float(saldo)
        self._lock = RLock()

    def setor(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah setoran harus lebih dari nol.")
        with self._lock:
            self.saldo += jumlah
            return self.saldo

    def tarik(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah penarikan harus lebih dari nol.")
        with self._lock:
            if self.saldo < jumlah:
                raise ValueError(f"Saldo {self.nama} tidak mencukupi.")
            self.saldo -= jumlah
            return self.saldo

    def transfer(self, rekening_tujuan, jumlah):
        if rekening_tujuan is self:
            raise ValueError("Rekening sumber dan tujuan tidak boleh sama.")
        if jumlah <= 0:
            raise ValueError("Jumlah transfer harus lebih dari nol.")

        # Kunci kedua rekening dengan urutan konsisten untuk mencegah
        # dua transfer berlawanan arah saling menunggu.
        rekening_terkunci = sorted(
            [self, rekening_tujuan], key=id
        )
        with ExitStack() as stack:
            for rekening in rekening_terkunci:
                stack.enter_context(rekening._lock)

            # Kedua method ini mengunci kembali objeknya sendiri.
            # RLock mengizinkan penguncian ulang oleh thread yang sama.
            self.tarik(jumlah)
            rekening_tujuan.setor(jumlah)

        return True

### Pengujian transaksi
Saldo awal rekening sumber adalah Rp1.000.000 dan rekening tujuan Rp500.000. Pengujian meliputi setoran, penarikan, transfer berhasil, dan penolakan penarikan yang melebihi saldo.

In [2]:
rekening_a = RekeningBank("A", 1_000_000)
rekening_b = RekeningBank("B", 500_000)

print("Saldo awal:")
print(rekening_a.nama, rekening_a.saldo)
print(rekening_b.nama, rekening_b.saldo)

rekening_a.setor(200_000)
rekening_a.tarik(100_000)
rekening_a.transfer(rekening_b, 300_000)

print("\\nSaldo akhir setelah setor, tarik, dan transfer:")
print(rekening_a.nama, rekening_a.saldo)
print(rekening_b.nama, rekening_b.saldo)

try:
    rekening_a.tarik(2_000_000)
except ValueError as error:
    print("\\nPengujian saldo tidak cukup:", error)

Saldo awal:
A 1000000.0
B 500000.0
\nSaldo akhir setelah setor, tarik, dan transfer:
A 800000.0
B 800000.0
\nPengujian saldo tidak cukup: Saldo A tidak mencukupi.


### Pengujian transfer bersamaan
Dua thread melakukan transfer dengan arah berlawanan. Penguncian berdasarkan urutan objek membuat keduanya mengambil lock dalam urutan yang sama, sehingga pola tunggu melingkar dapat dihindari. Jumlah total saldo seharusnya tetap.

In [3]:
akun_x = RekeningBank("X", 10_000)
akun_y = RekeningBank("Y", 10_000)

def transfer_berulang(sumber, tujuan, jumlah, banyak):
    for _ in range(banyak):
        sumber.transfer(tujuan, jumlah)

t1 = Thread(target=transfer_berulang, args=(akun_x, akun_y, 1, 500))
t2 = Thread(target=transfer_berulang, args=(akun_y, akun_x, 1, 500))

t1.start()
t2.start()
t1.join()
t2.join()

print("Saldo X:", akun_x.saldo)
print("Saldo Y:", akun_y.saldo)
print("Total saldo:", akun_x.saldo + akun_y.saldo)

Saldo X: 10000.0
Saldo Y: 10000.0
Total saldo: 20000.0


## 2. Deadlock dengan 3 Lock dan 3 Thread

### Skenario circular wait
Tiga lock (`Lock-A`, `Lock-B`, dan `Lock-C`) digunakan oleh tiga thread dengan pola berikut:

- Thread 1 memegang Lock-A lalu menunggu Lock-B.
- Thread 2 memegang Lock-B lalu menunggu Lock-C.
- Thread 3 memegang Lock-C lalu menunggu Lock-A.

Ketiga thread telah memegang satu lock dan saling menunggu lock berikutnya. Kondisi ini membentuk circular wait sehingga tidak ada thread yang dapat melanjutkan.

Simulasi dijalankan di proses anak (`subprocess`) dan dibatasi dengan `timeout`. Dengan demikian, proses utama notebook tidak ikut macet ketika deadlock benar-benar terjadi.

In [4]:
import subprocess
import sys
import textwrap

program_deadlock = textwrap.dedent(r'''
    import threading
    import time

    lock_a = threading.Lock()
    lock_b = threading.Lock()
    lock_c = threading.Lock()
    barrier = threading.Barrier(3)

    def worker(nama, lock_pertama, lock_kedua):
        with lock_pertama:
            print(f"{nama} memegang lock pertama", flush=True)
            barrier.wait()
            print(f"{nama} menunggu lock kedua", flush=True)
            lock_kedua.acquire()
            print(f"{nama} berhasil mendapat kedua lock", flush=True)
            lock_kedua.release()

    threads = [
        threading.Thread(target=worker, args=("Thread 1", lock_a, lock_b)),
        threading.Thread(target=worker, args=("Thread 2", lock_b, lock_c)),
        threading.Thread(target=worker, args=("Thread 3", lock_c, lock_a)),
    ]

    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
''')

try:
    hasil = subprocess.run(
        [sys.executable, "-u", "-c", program_deadlock],
        capture_output=True,
        text=True,
        timeout=3
    )
    print(hasil.stdout)
    print("Proses selesai tanpa timeout.")
except subprocess.TimeoutExpired as error:
    print("Output sebelum timeout:")
    if error.stdout:
        output = error.stdout.decode() if isinstance(error.stdout, bytes) else error.stdout
        print(output)
    print("TIMEOUT: proses tidak selesai dalam 3 detik.")
    print("Kesimpulan: simulasi menunjukkan deadlock (ketiga thread saling menunggu).")

Output sebelum timeout:
TIMEOUT: proses tidak selesai dalam 3 detik.
Kesimpulan: simulasi menunjukkan deadlock (ketiga thread saling menunggu).


### Perbaikan: urutan lock konsisten
Pencegahan dilakukan dengan menetapkan satu urutan global untuk mengambil lock. Semua thread harus mengambil lock berdasarkan urutan yang sama, bukan berdasarkan kebutuhan lokal masing-masing. Dengan cara ini, circular wait diputus.

In [5]:
import threading

lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()

# Urutan global yang wajib diikuti semua thread.
urutan_lock = [lock_a, lock_b, lock_c]
counter = {"nilai": 0}

def pekerjaan_aman(nama, jumlah_iterasi=100):
    for _ in range(jumlah_iterasi):
        with urutan_lock[0]:
            with urutan_lock[1]:
                with urutan_lock[2]:
                    counter["nilai"] += 1

threads_aman = [
    threading.Thread(target=pekerjaan_aman, args=(f"Thread {i}",))
    for i in range(3)
]

for thread in threads_aman:
    thread.start()
for thread in threads_aman:
    thread.join(timeout=3)

print("Nilai counter:", counter["nilai"])
print("Semua thread selesai:", all(not t.is_alive() for t in threads_aman))

Nilai counter: 300
Semua thread selesai: True


**Analisis:** Deadlock terjadi karena empat kondisi Coffman dapat terbentuk, khususnya mutual exclusion, hold and wait, no preemption, dan circular wait. Pada skenario ini, setiap thread mempertahankan lock yang telah diperoleh sambil menunggu lock milik thread lain. Penggunaan urutan lock global menghilangkan circular wait. Jika semua thread selalu meminta lock dari urutan terendah ke tertinggi, siklus tunggu tidak dapat terbentuk.

## 3. Bounded Buffer dengan Kapasitas Berbeda

### Tujuan dan rancangan eksperimen
Eksperimen mengulang producer–consumer untuk `UKURAN_BUFFER = 1, 3, 10`. Pada setiap percobaan terdapat **2 producer dan 2 consumer**. Masing-masing producer menghasilkan 8 item. `queue.Queue(maxsize=...)` digunakan sebagai bounded buffer yang aman untuk thread.

Producer akan menunggu ketika buffer penuh, sedangkan consumer akan menunggu ketika buffer kosong. Dua objek sentinel dikirim setelah seluruh producer selesai agar kedua consumer berhenti dengan tertib.

In [6]:
from queue import Queue
from threading import Thread, Lock
from collections import Counter
import time

def eksperimen_buffer(ukuran_buffer, item_per_producer=8):
    buffer = Queue(maxsize=ukuran_buffer)
    catatan = []
    lock_catatan = Lock()
    waktu_tunggu_put = []
    waktu_tunggu_get = []
    jumlah_producer = 2
    jumlah_consumer = 2
    total_item = jumlah_producer * item_per_producer
    SENTINEL = object()

    def producer(nomor):
        for i in range(item_per_producer):
            item = (nomor, i + 1)
            mulai = time.perf_counter()
            buffer.put(item)
            durasi = time.perf_counter() - mulai
            with lock_catatan:
                waktu_tunggu_put.append(durasi)
                catatan.append(("P", nomor, item))
            time.sleep(0.002)

    def consumer(nomor):
        while True:
            mulai = time.perf_counter()
            item = buffer.get()
            durasi = time.perf_counter() - mulai
            with lock_catatan:
                waktu_tunggu_get.append(durasi)

            if item is SENTINEL:
                buffer.task_done()
                break

            with lock_catatan:
                catatan.append(("C", nomor, item))
            time.sleep(0.003)
            buffer.task_done()

    consumers = [
        Thread(target=consumer, args=(i + 1,))
        for i in range(jumlah_consumer)
    ]
    producers = [
        Thread(target=producer, args=(i + 1,))
        for i in range(jumlah_producer)
    ]

    for thread in consumers:
        thread.start()
    for thread in producers:
        thread.start()

    for thread in producers:
        thread.join()

    # Sentinel dimasukkan setelah producer selesai.
    for _ in range(jumlah_consumer):
        buffer.put(SENTINEL)

    for thread in consumers:
        thread.join()

    diproduksi = [e for e in catatan if e[0] == "P"]
    dikonsumsi = [e for e in catatan if e[0] == "C"]
    item_dikonsumsi = [e[2] for e in dikonsumsi]

    return {
        "kapasitas": ukuran_buffer,
        "jumlah_diproduksi": len(diproduksi),
        "jumlah_dikonsumsi": len(dikonsumsi),
        "item_unik_dikonsumsi": len(set(item_dikonsumsi)),
        "waktu_put_di_atas_1ms": sum(x > 0.001 for x in waktu_tunggu_put),
        "waktu_get_di_atas_1ms": sum(x > 0.001 for x in waktu_tunggu_get),
        "catatan": catatan
    }

### Menjalankan tiga variasi kapasitas
Ringkasan memeriksa apakah seluruh item berhasil diproduksi dan dikonsumsi tepat satu kali. Kolom operasi menunggu lebih dari 1 ms merupakan indikasi sederhana waktu tunggu pada eksekusi tersebut, bukan ukuran performa yang mutlak.

In [7]:
hasil_buffer = {}
for kapasitas in [1, 3, 10]:
    hasil_buffer[kapasitas] = eksperimen_buffer(kapasitas)

for kapasitas, hasil in hasil_buffer.items():
    print(f"\\nKapasitas buffer: {kapasitas}")
    print("Diproduksi:", hasil["jumlah_diproduksi"])
    print("Dikonsumsi:", hasil["jumlah_dikonsumsi"])
    print("Item unik dikonsumsi:", hasil["item_unik_dikonsumsi"])
    print("Operasi put > 1 ms:", hasil["waktu_put_di_atas_1ms"])
    print("Operasi get > 1 ms:", hasil["waktu_get_di_atas_1ms"])

\nKapasitas buffer: 1
Diproduksi: 16
Dikonsumsi: 16
Item unik dikonsumsi: 16
Operasi put > 1 ms: 8
Operasi get > 1 ms: 0
\nKapasitas buffer: 3
Diproduksi: 16
Dikonsumsi: 16
Item unik dikonsumsi: 16
Operasi put > 1 ms: 3
Operasi get > 1 ms: 0
\nKapasitas buffer: 10
Diproduksi: 16
Dikonsumsi: 16
Item unik dikonsumsi: 16
Operasi put > 1 ms: 0
Operasi get > 1 ms: 0


### Mengamati pola interleaving
Urutan berikut memperlihatkan sebagian kejadian producer (P) dan consumer (C). Karena thread berjalan konkuren, urutan detail dapat berubah setiap kali notebook dijalankan.

In [8]:
for kapasitas, hasil in hasil_buffer.items():
    print(f"\\nKapasitas {kapasitas} — 20 kejadian pertama:")
    for kejadian in hasil["catatan"][:20]:
        jenis, nomor_thread, item = kejadian
        if jenis == "P":
            print(f"Producer {nomor_thread} menghasilkan {item}")
        else:
            print(f"Consumer {nomor_thread} mengambil {item}")

\nKapasitas 1 — 20 kejadian pertama:
Producer 1 menghasilkan (1, 1)
Consumer 1 mengambil (1, 1)
Producer 2 menghasilkan (2, 1)
Consumer 2 mengambil (2, 1)
Producer 1 menghasilkan (1, 2)
Consumer 1 mengambil (1, 2)
Producer 2 menghasilkan (2, 2)
Consumer 2 mengambil (2, 2)
Producer 1 menghasilkan (1, 3)
Consumer 1 mengambil (1, 3)
Producer 2 menghasilkan (2, 3)
Consumer 2 mengambil (2, 3)
Producer 1 menghasilkan (1, 4)
Consumer 1 mengambil (1, 4)
Producer 2 menghasilkan (2, 4)
Consumer 2 mengambil (2, 4)
Producer 1 menghasilkan (1, 5)
Consumer 1 mengambil (1, 5)
Producer 2 menghasilkan (2, 5)
Consumer 2 mengambil (2, 5)
\nKapasitas 3 — 20 kejadian pertama:
Producer 1 menghasilkan (1, 1)
Consumer 1 mengambil (1, 1)
Producer 2 menghasilkan (2, 1)
Consumer 2 mengambil (2, 1)
Producer 1 menghasilkan (1, 2)
Producer 2 menghasilkan (2, 2)
Consumer 1 mengambil (1, 2)
Consumer 2 mengambil (2, 2)
Producer 1 menghasilkan (1, 3)
Producer 2 menghasilkan (2, 3)
Consumer 1 mengambil (1, 3)
Producer 1

### Analisis pengaruh kapasitas buffer
- **Kapasitas 1:** buffer hampir selalu menjadi titik sinkronisasi langsung. Producer lebih mudah tertahan ketika satu slot sedang terisi, sehingga pergantian producer–consumer cenderung lebih rapat.
- **Kapasitas 3:** tersedia ruang penyangga lebih besar. Producer dapat memasukkan beberapa item sebelum harus menunggu consumer, sehingga interleaving dapat menunjukkan beberapa produksi berurutan.
- **Kapasitas 10:** kapasitas cukup untuk menampung seluruh item dari satu producer, bahkan mendekati seluruh hasil produksi. Producer umumnya lebih jarang tertahan oleh buffer, sehingga consumer dapat mengambil item secara berkelompok setelah item tersedia.

Kapasitas yang lebih besar tidak menjamin urutan producer dan consumer menjadi tetap atau selalu lebih cepat. Pola aktual juga dipengaruhi kecepatan kerja, jeda (`sleep`), dan penjadwalan thread. Yang dijaga oleh bounded buffer adalah batas jumlah item yang dapat menunggu di dalam antrean.

## 4. Refleksi Singkat (maksimal 150 kata)

Menurut saya, **deadlock** paling berbahaya bagi aplikasi produksi karena dapat menghentikan proses penting secara permanen, misalnya transaksi pembayaran yang membutuhkan beberapa sumber daya sekaligus. Dampaknya bukan hanya keterlambatan, tetapi juga layanan dapat tidak merespons sampai proses dihentikan atau dipulihkan. **Starvation** juga berbahaya karena suatu thread dapat terus-menerus tidak memperoleh kesempatan menggunakan sumber daya, meskipun sistem masih berjalan. Sementara itu, **livelock** membuat thread tetap aktif tetapi tidak menghasilkan kemajuan yang berguna. Masalah yang paling sulit dideteksi saat testing adalah starvation, karena gejalanya dapat muncul tidak konsisten dan bergantung pada beban serta penjadwalan sistem. Pengujian singkat mungkin terlihat normal, padahal pada kondisi beban tinggi satu proses dapat terus tertunda. Oleh sebab itu, pengujian konkurensi perlu dilakukan berulang dengan variasi beban dan pemantauan waktu tunggu setiap proses.

## Kesimpulan

1. `RLock` diperlukan ketika sebuah method yang telah mengunci objek memanggil method lain pada objek yang sama dan method tersebut juga melakukan penguncian.
2. Deadlock tiga thread terbukti melalui pola circular wait dan dapat dicegah dengan urutan pengambilan lock yang konsisten.
3. Kapasitas bounded buffer memengaruhi seberapa sering producer dan consumer harus menunggu serta pola interleaving, tetapi tidak mengubah jumlah item yang seharusnya diproses.

